# 第5课：联合模拟2026年July与Harvest Futures价格

本课把第3课的产量情景与第4课的**成对价格Residual**合并，生成10,000个共同情景。  
本课只做到December corn futures价格；暂时不加入basis、cash price、套保合约或利润。

请从上到下逐格运行。

## 0. 本课到底在模拟什么？

每个情景 $i$ 都有一组相互匹配的结果：

1. 随机抽一个历史July PDSI，得到7月天气下的产量预测；
2. 随机抽一个历史产量Residual，得到最终产量；
3. 随机抽**同一年**的July和Harvest价格Residual；
4. 从2026年3月期货价格 $F_0=\$4.70/bu$ 出发，得到July和Harvest期货价格。

这样以后比较不同套保策略时，所有策略都面对完全相同的10,000个未来情景，这叫 **common random numbers**。

## 1. 四个基础公式

### 产量部分

$$\widehat{Y}_{July,i}=\beta_0+\beta_1(30)+\beta_2PDSI_i$$

$$Y_i=\max(0,\widehat{Y}_{July,i}+e^Y_i)$$

### 价格部分

$$F_{July,i}=\max\left(1.50,F_0+\alpha_J+\gamma_J(\widehat{Y}_{July,i}-Y_{trend})+e^J_i\right)$$

$$F_{Harvest,i}=\max\left(1.50,F_0+\alpha_H+\gamma_H(Y_i-Y_{trend})+e^H_i\right)$$

注意：两个价格变化都定义为**相对3月价格 $F_0$ 的累计变化**，所以Harvest不是在July价格上再加一次变化。

## 2. 导入工具并锁定设置

`RANDOM_SEED`让每次运行得到同一组随机数，方便老师与组员复核。  
`PRICE_FLOOR`是模型假设，不是数据事实；它防止极端Residual产生不合理的负价格。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_SIMULATIONS = 10_000
RANDOM_SEED = 8_122_026
FORECAST_TREND_INDEX = 30
F0 = 4.70
PRICE_FLOOR = 1.50

print('模拟次数:', N_SIMULATIONS)
print('随机种子:', RANDOM_SEED)
print('2026年3月December futures起点: $', F0, '/bu')
print('价格下限假设: $', PRICE_FLOOR, '/bu')

## 3. 输入第2课与第4课得到的模型参数

这些参数来自1996–2025年30个crop years的校准。  
2026年的`trend_index = 2026 - 1996 = 30`。

In [ ]:
# Yield model: Trend + July PDSI
YIELD_INTERCEPT = 138.3603607964906
YIELD_TREND_BETA = 2.318714127208453
YIELD_PDSI_BETA = 1.907873690521471

# Trend-only model: used as the price-signal benchmark
TREND_ONLY_INTERCEPT = 140.9268817204301
TREND_ONLY_BETA = 2.310789766407122
TREND_YIELD_2026 = TREND_ONLY_INTERCEPT + TREND_ONLY_BETA * FORECAST_TREND_INDEX

# Price-change models
JULY_PRICE_INTERCEPT = -0.05375000000000204
JULY_SIGNAL_BETA = -0.019741301483521958
HARVEST_PRICE_INTERCEPT = -0.1506666666666673
HARVEST_SURPRISE_BETA = -0.026009894449960193

print(f'2026 trend-only benchmark yield = {TREND_YIELD_2026:.6f} bu/acre')
print('July price signal beta:', JULY_SIGNAL_BETA)
print('Harvest yield-surprise beta:', HARVEST_SURPRISE_BETA)

## 4. 建立历史Shock Library

下面30行是前面课程计算好的历史随机冲击：

- `july_pdsi`：July天气状态；
- `yield_residual`：实际产量减去Trend + PDSI模型预测；
- `july_price_residual`与`harvest_price_residual`：价格模型没有解释的部分。

产量Residual和天气分开抽样，是本项目的建模假设；它允许组合出历史上没有完全出现过、但由历史冲击构成的新情景。

In [ ]:
shock_library = pd.DataFrame({
    'year': list(range(1996, 2026)),
    'july_pdsi': [
        1.51, -0.36, 2.39, 3.10, 1.11, -0.37, 0.06, 0.78, 1.47, -0.21,
        -2.39, 1.27, 6.45, 3.98, 6.69, -0.23, -3.56, -0.68, 2.27, 3.18,
        3.50, 2.06, 1.53, 4.70, -0.36, -1.43, -0.89, -1.99, 2.43, 2.54
    ],
    'yield_residual': [
        -3.2412500691780224, -1.992240395111338, -2.5576071712538067,
        -2.2309116187325344, -5.752957101803247, -3.2480181670399304,
        10.612882018827406, 0.9204988344434925, 21.285351860775226,
        14.17186553364283, 9.012316051771194, 4.710784217254144,
        -7.490715626855547, 4.903018261524068, -18.5860335669976,
        -0.7022617557974513, -31.667756493569414, -12.4811468494797,
        -6.428088363726488, 3.5170324506905217, 11.58779874251519,
        11.016422729657648, 3.7088816584255824, -2.657792067735926,
        -16.32266532090574, 10.400045400743778, 3.0510794806537263,
        3.83102641301889, 3.079510573705562, -0.4490696594602639
    ],
    'july_price_residual': [
        0.6149560403495117, -0.385319033625754, -0.3940868447081854,
        -0.2321890316134258, -0.52698377475474, -0.2350699239403515,
        0.1362819944336285, -0.2364435533744417, -0.4302990184740243,
        0.2965820503969517, -0.0203688355906054, -0.5948624888305076,
        1.2603930006995283, -0.3949804191059779, 0.2647452134229091,
        0.7192673953764686, 1.929003013214289, -0.4798684896049224,
        -0.8061035187439403, 0.4033270762942307, 0.0330359645993351,
        -0.0485436281975942, -0.4308490631519948, 0.6537019678467413,
        -0.4392209781259594, 0.7856350756724253, -0.1663699758847157,
        -0.700143839377844, -0.4060129212327776, -0.1692134539682568
    ],
    'harvest_price_residual': [
        -0.4129612179492374, -0.0155646158695341, -0.5385987526401107,
        -0.1771625727605671, -0.4348154429306652, -0.4878990519510425,
        0.5241657557779833, 0.1605029911579246, -0.2528629399633283,
        0.0114545065166931, 0.9217818474466736, -0.1732720782238226,
        -1.518375476144119, 0.4316200704351839, 1.642858361315524,
        0.5523242245449478, 0.9293745208760434, -1.565961726895328,
        -0.9969266025161831, -0.0303914781370378, 0.3156139628922276,
        -0.0529993294780311, -0.1191620940980894, 0.2027542968815342,
        -0.1860568844879279, 1.3911068677407004, 0.8544638920205627,
        -0.8521296114497754, -0.1571340648704713, 0.0342526427592721
    ]
})

print('Shock library行数:', len(shock_library))
print(shock_library.head().round(4).to_string(index=False))

## 5. 为什么价格Residual必须成对抽？

同一年7月和收获期的价格冲击可能有关。如果分别独立抽，会破坏这种时间相关性。  
因此我们抽一个`price_row`，同时拿走该年的 $(e^J,e^H)$。这正面回应教授要求：July和Harvest的价格过程必须被清楚地区分并正确追踪。

In [ ]:
historical_pair_corr = shock_library[
    ['july_price_residual', 'harvest_price_residual']
].corr().iloc[0, 1]

print(f'历史成对价格Residual相关系数 = {historical_pair_corr:.6f}')
assert len(shock_library) == 30
assert np.isclose(historical_pair_corr, 0.34724696594224497)
print('检查通过：30组历史shock存在，价格Residual配对关系正确。')

## 6. 抽取10,000组随机来源年份

随机数的顺序也必须锁定：

1. 抽天气年份；
2. 抽产量Residual年份；
3. 抽价格Residual年份。

三个来源可以不同，但July和Harvest价格Residual始终来自同一个`price_source_year`。

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
n_hist = len(shock_library)

weather_rows = rng.integers(0, n_hist, size=N_SIMULATIONS)
yield_rows = rng.integers(0, n_hist, size=N_SIMULATIONS)
price_rows = rng.integers(0, n_hist, size=N_SIMULATIONS)

pdsi_draw = shock_library.loc[weather_rows, 'july_pdsi'].to_numpy()
yield_residual_draw = shock_library.loc[yield_rows, 'yield_residual'].to_numpy()
july_price_residual_draw = shock_library.loc[price_rows, 'july_price_residual'].to_numpy()
harvest_price_residual_draw = shock_library.loc[price_rows, 'harvest_price_residual'].to_numpy()

weather_source_year = shock_library.loc[weather_rows, 'year'].to_numpy()
yield_source_year = shock_library.loc[yield_rows, 'year'].to_numpy()
price_source_year = shock_library.loc[price_rows, 'year'].to_numpy()

print('抽样完成。前5个weather source years:', weather_source_year[:5])
print('前5个yield residual source years:', yield_source_year[:5])
print('前5个paired price source years:', price_source_year[:5])

## 7. 先生成July产量预测与Final Yield

`july_yield_forecast`只使用7月时已经知道的PDSI。  
`final_yield`再加入直到收获才完全观察到的产量Residual。

In [ ]:
july_yield_forecast = (
    YIELD_INTERCEPT
    + YIELD_TREND_BETA * FORECAST_TREND_INDEX
    + YIELD_PDSI_BETA * pdsi_draw
)

final_yield = np.maximum(0.0, july_yield_forecast + yield_residual_draw)

print(f'July yield forecast平均值 = {july_yield_forecast.mean():.3f} bu/acre')
print(f'Final yield平均值 = {final_yield.mean():.3f} bu/acre')
print(f'Final yield标准差 = {final_yield.std(ddof=1):.3f} bu/acre')

## 8. 把产量信息转成价格Signal

价格模型使用“相对于2026 trend-only产量”的偏离：

$$JulySignal_i=\widehat{Y}_{July,i}-Y_{trend}$$

$$HarvestSurprise_i=Y_i-Y_{trend}$$

价格斜率为负，意思是模拟中产量高于趋势时，期货价格倾向更低；反之亦然。

In [ ]:
july_yield_signal = july_yield_forecast - TREND_YIELD_2026
final_yield_surprise = final_yield - TREND_YIELD_2026

print(pd.DataFrame({
    'July signal': july_yield_signal,
    'Harvest surprise': final_yield_surprise
}).describe().round(3).to_string())

## 9. 生成July与Harvest Futures

模型先计算从3月到目标日期的累计变化，再加到同一个 $F_0=4.70$ 上：

$$\Delta F_{July,i}=\alpha_J+\gamma_JJulySignal_i+e^J_i$$

$$\Delta F_{Harvest,i}=\alpha_H+\gamma_HHarvestSurprise_i+e^H_i$$

最后用`np.maximum`实施 $\$1.50/bu$ 的价格下限。

In [ ]:
july_futures_change = (
    JULY_PRICE_INTERCEPT
    + JULY_SIGNAL_BETA * july_yield_signal
    + july_price_residual_draw
)

harvest_futures_change = (
    HARVEST_PRICE_INTERCEPT
    + HARVEST_SURPRISE_BETA * final_yield_surprise
    + harvest_price_residual_draw
)

july_futures = np.maximum(PRICE_FLOOR, F0 + july_futures_change)
harvest_futures = np.maximum(PRICE_FLOOR, F0 + harvest_futures_change)

print('July futures生成完成。')
print('Harvest futures生成完成。')

## 10. 组合成共同情景表

每一行代表一个可能的2026未来。下一课加入basis时，会继续在这张表上增加列。

In [ ]:
scenarios = pd.DataFrame({
    'scenario_id': np.arange(1, N_SIMULATIONS + 1),
    'weather_source_year': weather_source_year,
    'yield_residual_source_year': yield_source_year,
    'price_residual_source_year': price_source_year,
    'july_pdsi': pdsi_draw,
    'yield_residual_bu_per_acre': yield_residual_draw,
    'july_price_residual_usd_per_bushel': july_price_residual_draw,
    'harvest_price_residual_usd_per_bushel': harvest_price_residual_draw,
    'preseason_expected_yield_bu_per_acre': TREND_YIELD_2026,
    'july_yield_forecast_bu_per_acre': july_yield_forecast,
    'final_yield_bu_per_acre': final_yield,
    'preseason_futures_usd_per_bushel': F0,
    'july_futures_usd_per_bushel': july_futures,
    'harvest_futures_usd_per_bushel': harvest_futures,
})

print(scenarios.head(10).round(4).to_string(index=False))

## 11. 必须通过的模型检查

这些`assert`不是多余代码。它们是在正式作业中证明：

- 恰好生成10,000行；
- 没有缺失值；
- 情景编号没有重复；
- 3月基准价格始终是$4.70；
- July和Harvest价格没有跌破设定下限；
- 同一个price source year同时控制两段价格Residual。

In [ ]:
assert len(scenarios) == N_SIMULATIONS
assert scenarios['scenario_id'].is_unique
assert scenarios.isna().sum().sum() == 0
assert np.allclose(scenarios['preseason_futures_usd_per_bushel'], F0)
assert (scenarios['july_futures_usd_per_bushel'] >= PRICE_FLOOR).all()
assert (scenarios['harvest_futures_usd_per_bushel'] >= PRICE_FLOOR).all()

# 由price_rows重新查回同年Residual，验证成对抽样没有错位
assert np.allclose(
    july_price_residual_draw,
    shock_library.loc[price_rows, 'july_price_residual'].to_numpy()
)
assert np.allclose(
    harvest_price_residual_draw,
    shock_library.loc[price_rows, 'harvest_price_residual'].to_numpy()
)

print('全部模型检查通过。')

## 12. 查看价格分布

我们报告均值、标准差、5th percentile、中位数与95th percentile。  
后面选择策略时不能只看均值，还要一起看收入波动和downside risk。

In [ ]:
def distribution_summary(series):
    return pd.Series({
        'Mean': series.mean(),
        'Std Dev': series.std(ddof=1),
        'P5': series.quantile(0.05),
        'Median': series.median(),
        'P95': series.quantile(0.95),
        'Minimum': series.min(),
        'Maximum': series.max(),
        'Floor Count': int((series <= PRICE_FLOOR + 1e-12).sum()),
    })

price_summary = pd.DataFrame({
    'July Futures': distribution_summary(scenarios['july_futures_usd_per_bushel']),
    'Harvest Futures': distribution_summary(scenarios['harvest_futures_usd_per_bushel']),
}).T

print(price_summary.round(4).to_string())

## 13. 画出两个价格分布

图形让我们看到分布的中心、宽度和尾部，而不仅是一张数字表。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].hist(july_futures, bins=35, color='#2F75B5', edgecolor='white')
axes[0].axvline(july_futures.mean(), color='black', linestyle='--', label='Mean')
axes[0].set_title('July December Futures')
axes[0].set_xlabel('USD per bushel')
axes[0].set_ylabel('Number of simulations')
axes[0].legend()

axes[1].hist(harvest_futures, bins=35, color='#70AD47', edgecolor='white')
axes[1].axvline(harvest_futures.mean(), color='black', linestyle='--', label='Mean')
axes[1].set_title('Harvest December Futures')
axes[1].set_xlabel('USD per bushel')
axes[1].legend()

plt.tight_layout()
plt.show()

## 14. 检查情景内部关系

下面三个相关系数不是额外的回归，它们只是在检查模拟结果是否符合模型方向：

- Final yield和Harvest futures应该负相关；
- July forecast和July futures应该弱负相关；
- July和Harvest futures因成对Residual而呈正相关。

In [ ]:
relationship_checks = pd.Series({
    'Corr(July futures, Harvest futures)': scenarios['july_futures_usd_per_bushel'].corr(
        scenarios['harvest_futures_usd_per_bushel']
    ),
    'Corr(July yield forecast, July futures)': scenarios['july_yield_forecast_bu_per_acre'].corr(
        scenarios['july_futures_usd_per_bushel']
    ),
    'Corr(Final yield, Harvest futures)': scenarios['final_yield_bu_per_acre'].corr(
        scenarios['harvest_futures_usd_per_bushel']
    ),
})

print(relationship_checks.round(4).to_string())

## 15. 可重复性检查

下面的数值来自锁定的seed和参数。如果你没有修改上面的数据或公式，应该完全通过。

In [ ]:
expected = {
    'july_mean': 4.637581762133069,
    'july_std': 0.6069642578032922,
    'harvest_mean': 4.541299446091132,
    'harvest_std': 0.7837335500240272,
    'price_corr': 0.336098877666771,
}

actual = {
    'july_mean': july_futures.mean(),
    'july_std': july_futures.std(ddof=1),
    'harvest_mean': harvest_futures.mean(),
    'harvest_std': harvest_futures.std(ddof=1),
    'price_corr': np.corrcoef(july_futures, harvest_futures)[0, 1],
}

for key in expected:
    assert np.isclose(actual[key], expected[key], atol=1e-12), (key, actual[key], expected[key])

print('可重复性检查通过。')
print(pd.DataFrame({'Expected': expected, 'Actual': actual}).round(6).to_string())

## 16. 保存本课结果

运行后会在当前目录建立`lesson_05_outputs`，保存CSV与JSON摘要。  
CSV保留全部公式输入与来源年份，方便下一课直接加入basis和cash price。

In [ ]:
OUTPUT_DIR = Path.cwd() / 'lesson_05_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

scenario_path = OUTPUT_DIR / 'futures_scenarios_10000.csv'
summary_path = OUTPUT_DIR / 'step_05_summary.json'

scenarios.to_csv(scenario_path, index=False)

summary_for_json = {
    'n_simulations': N_SIMULATIONS,
    'random_seed': RANDOM_SEED,
    'preseason_futures_usd_per_bushel': F0,
    'price_floor_usd_per_bushel': PRICE_FLOOR,
    'july_futures': {k: float(v) for k, v in distribution_summary(
        scenarios['july_futures_usd_per_bushel']
    ).items()},
    'harvest_futures': {k: float(v) for k, v in distribution_summary(
        scenarios['harvest_futures_usd_per_bushel']
    ).items()},
    'july_harvest_price_correlation': float(
        scenarios['july_futures_usd_per_bushel'].corr(
            scenarios['harvest_futures_usd_per_bushel']
        )
    ),
}

summary_path.write_text(
    json.dumps(summary_for_json, indent=2),
    encoding='utf-8'
)

print('已保存:', scenario_path)
print('已保存:', summary_path)

## 17. 本课结论与限制

完成本课后，我们有了10,000个可以复核的联合情景，但**还不能评价哪种套保策略最好**，因为尚未加入basis、cash price、期货合约数量、交易价格和成本。

必须保留的限制说明：

1. 历史期货输入是Barchart/TradingCharts daily close，不是官方CME settlement；
2. 2026起点是2026-03-02的ZCZ26 daily close，$4.70/bu；
3. July价格模型解释力很弱（$R^2\approx0.022$）；最终必须做`July beta = 0`的robustness check；
4. $1.50/bu$价格下限、独立抽取天气与产量Residual都是显式模型假设；
5. 所有策略必须使用本课生成的同一组情景，才是公平比较。

**下一课：模拟basis，并用 $CashPrice=HarvestFutures+Basis$ 得到收获期现金价格。**